# Reading CIF Files

CrystalBuilder includes some rudimentary methods to get atom and unit cell information from CIF files. These files are used extensively in crystallography and use a [syntax specified by the IUCr](https://www.iucr.org/resources/cif/cif2)

In [1]:
import crystalbuilder as cb

geo = cb.geometry
lattice = cb.lattice
viewer = cb.newviewer

import crystalbuilder.utilities as cbutils #The cif-reader module is included in utilities.

DEBUG:crystalbuilder.bilbao:Bilbao is logging.
DEBUG:trimesh.util:searching for blender in: C:\WINDOWS;C:\Users\Brandon\miniconda3\envs\crystalbuilder-dev\Scripts;C:\WINDOWS\System32\Wbem;C:\Program Files\PowerShell\7;C:\Users\Brandon\miniconda3\Scripts;C:\Program Files (x86)\NVIDIA Corporation\PhysX\Common;c:\Users\Brandon\miniconda3\envs\crystalbuilder-dev;C:\Program Files\Microsoft Visual Studio\2022\Community\VC\Tools\MSVC\14.40.33807\bin\Hostx64\x64\cl.exe;C:\Program Files (x86)\IntelSWToolsMPI\compilers_and_libraries_2020.4.321\windows\mpi\intel64\bin;C:\Users\Brandon\miniconda3\envs\crystalbuilder-dev;C:\Program Files\PuTTY;C:\Program Files\NVIDIA GPU Computing Toolkit\CUDA\v12.4\libnvvp;C:\Users\Brandon\miniconda3\envs\crystalbuilder-dev\bin;C:\WINDOWS\System32\WindowsPowerShell\v1.0;C:\Users\Brandon\AppData\Local\GitHubDesktop\bin;C:\Users\Brandon\AppData\Roaming\npm;C:\Program Files\Calibre2;C:\Users\Brandon\miniconda3\condabin;C:\texlive\2024\bin;C:\Program Files (x86)\Windo

IndentationError: unindent does not match any outer indentation level (newviewer.py, line 220)

I've downloaded and included an example CIF file with the same space group as our good friend diamond Group Number 227, Fd-3m with origin choice 2. This comes from a [1997 paper by J-J Maguer et al.](https://www.sciencedirect.com/science/article/abs/pii/S0022459696971455) and was obtained from the [Crystallography Open Database](https://www.crystallography.net/cod/1000444.html)

In [2]:
cif_filename = "1000444.cif"
cif_file = cbutils.CIF_file(cif_filename)

We can check some basic parameters of our imported CIF file by looking at the parsed dictionary

In [3]:
cif_file.dictionary

{'Space Group': 227,
 'cell_angle_alpha': 90.0,
 'cell_angle_beta': 90.0,
 'cell_angle_gamma': 90.0,
 'cell_formula_units_Z': 16.0,
 'cell_length_a': 15.339,
 'cell_length_b': 15.339,
 'cell_length_c': 15.339,
 'cell_volume': 3609.0,
 'positions': {'Yb1': [0.375, 0.375, 0.053],
  'F1': [0.0, 0.8761, 0.1239],
  'F2': [0.2131, 0.2131, 0.2131],
  'F3': [0.0524, 0.0524, 0.0524],
  'K1': [0.5, 0.5, 0.5],
  'O2': [0.375, 0.375, 0.251]}}

Here you can see the unit cell information such as angles and sizes, as well as the positions of atoms. This structure is potassium triytterbium decafluoride hydrate-$\delta$ and has the formula F<sub>10</sub>H<sub>2</sub>KOYb<sub>3</sub>

Next, we can use the CIF_structure utility to create a model of the unit cell. One method of doing this is to create an instance of the CIF_structure class by passing the dictionary from above. 

In [4]:
crystal = cbutils.CIF_structure(cif_file.dictionary)

Then, we can take advantage of the CIF_structure functions to build the atoms as spheres. Here, we'll build them as identical atoms (i.e. no coloring or size changes to distinguish elements). The `_build_generic_atoms()` function accepts a radius argument and returns a list of crystalbuilder.geometry Spheres.

It's important to note at this stage that the different origin choices *are not yet* supported by the CIF building process. The origin choice will be whatever is default on Bilbao.

In [5]:
spheres = crystal._build_generic_atoms(radius=.05)

And we can look at these in the usual way, via a viewer window:

In [6]:
scene =viewer.visualize(spheres)
scene.show()

mode:vedo


Let's go a step further and create an entire crystal. CIF_structure also detemines the lattice vectors of the crystal and saves them as a `lattice_vectors` property. 

In [7]:
print(crystal.lattice_vectors)
lat = lattice.Lattice(*crystal.lattice_vectors, magnitude=[1,1,1]) #One annoyance for now is that you must unzip the list of vectors. This will be fixed eventually...

full_crystal = lat.tile_geogeometry(spheres, 2,1,1)


[array([1., 0., 0.]), array([0., 1., 0.]), array([0., 0., 1.])]


In [8]:
scene2 = viewer.visualize(full_crystal)
scene2.show()

mode:vedo
